# 信頼バンドの根探索 (`find_confidence_band`)

等間隔グリッドの代わりに、ブラケット＋幾何二分法でバンドの上下端を直接探す例。
最も広い 2σ で外側をブラケットし、各レベル (1σ/90%/2σ) の端を入れ子で詰める。

- `num_pseudo_data`: ブラケット段階の擬似データ数（粗くてよい）
- `n_pseudo_edge`: 端の二分法で使う擬似データ数（cutoff のノイズを抑えるため大きめ）
- `seed`: 固定すると各点が再現可能になり、二分法のジッターが減る

In [ ]:
from neutrino_analysis_band import NeutrinoAnalysis

a = NeutrinoAnalysis(background_scenario='flat', intervals='360',
                     GeV=0.32e16, solver='osqp')
a.optimize(a.data_vector)   # ベストフィットを先に求める (self.result を埋める)

In [ ]:
# 1 パラメータ (index=1) の 1σ/90%/2σ バンドを一度に求める
band = a.find_confidence_band(
    fixed_index=1,
    levels=(0.678, 0.90, 0.954),
    num_pseudo_data=20,    # ブラケット用 (粗い)
    n_pseudo_edge=200,     # 端の精緻化用 (大きいほど安定・遅い)
    step=1.5,              # ブラケットの拡大率
    rel_tol=0.03,          # 端の相対許容幅
    seed=42,
    n_jobs=1,
    verbose=True,
)
band

In [ ]:
# 物理単位での上下端を表に
import pandas as pd
rows = []
for lv, (lo, hi) in band['band_physical'].items():
    rows.append({'level': lv, 'lower': lo, 'upper': hi})
band_df = pd.DataFrame(rows)
print('best fit (phys) =', band['best_fit_physical'])
print('evaluations     =', band['n_evaluations'])
band_df

In [ ]:
# バンドを可視化 (ベストフィットを中心に各レベルの区間を横棒で表示)
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(8, 3))
colors = {0.678: 'C0', 0.90: 'C1', 0.954: 'C2'}
for i, lv in enumerate(band['levels']):
    lo, hi = band['band_physical'][lv]
    ax.hlines(i, lo, hi, color=colors.get(lv, 'k'), lw=6, alpha=0.6,
              label=f'{lv:.3f}')
ax.axvline(band['best_fit_physical'], color='red', ls='--', label='best fit')
ax.set_yticks(range(len(band['levels'])))
ax.set_yticklabels([f'{lv:.3f}' for lv in band['levels']])
ax.set_xscale('log')
ax.set_xlabel(r'$\Phi$ [cm$^{-2}$sec$^{-1}$]')
ax.set_ylabel('confidence level')
ax.set_title(f"Confidence band for parameter index {band['index']}")
ax.legend(loc='best', fontsize=8)
plt.tight_layout()
plt.show()

## 収束チェック (任意)

端の値は `n_pseudo_edge` に依存する（cutoff の Monte Carlo ノイズ）。
値を増やして端が安定するか確認するとよい。

In [ ]:
for npe in (100, 300):
    b = a.find_confidence_band(1, num_pseudo_data=20, n_pseudo_edge=npe,
                               seed=42, verbose=False)
    lo90, hi90 = b['band_physical'][0.90]
    print(f'n_pseudo_edge={npe:4d}  90% band = [{lo90:.4e}, {hi90:.4e}]  evals={b["n_evaluations"]}')